# PDF RAG System using Ollama and Llama 3.2

## Overview

This project implements a Retrieval-Augmented Generation (RAG) system
that allows users to ask questions about information contained in
multiple insurance-related PDF documents.

The system follows these major steps:

1. Load multiple PDF documents.
2. Extract text from the PDF files.
3. Split the extracted text into smaller chunks.
4. Convert the text chunks into vector embeddings.
5. Store the embeddings in ChromaDB.
6. Retrieve the most relevant chunks for a user's question.
7. Send the retrieved context to Llama 3.2 through Ollama.
8. Generate a grounded answer using only the retrieved information.
9. Display the source PDF and page number used for the answer.

## Technologies Used

- Python,Jupyter Notebook,PyPDF,Sentence Transformers,ChromaDB
- Ollama,Llama 3.2,NumPy,Requests

## RAG Architecture

PDF Documents
       ↓
PDF Text Extraction
       ↓
Text Cleaning
       ↓
Text Chunking
       ↓
Sentence Transformer Embeddings
       ↓
ChromaDB Vector Database
       ↓
User Question
       ↓
Question Embedding
       ↓
Similarity Search
       ↓
Relevant Document Chunks
       ↓
Llama 3.2 through Ollama
       ↓
Grounded Answer + Sources

## Objective

The objective of this project is to demonstrate how a local
Large Language Model can answer questions from a collection of
documents using Retrieval-Augmented Generation instead of relying
only on the model's pre-trained knowledge.

In [2]:
# ============================================================
# Cell 2: Import Required Libraries
# ============================================================

# Path is used for handling folders and file paths.
from pathlib import Path

# PyPDF is used to read and extract text from PDF documents.
from pypdf import PdfReader

# ChromaDB is used as our vector database.
import chromadb

# SentenceTransformer is used to create text embeddings.
from sentence_transformers import SentenceTransformer

# Requests is used to communicate with the local Ollama API.
import requests

# NumPy is used for numerical operations.
import numpy as np

# Display is used to display formatted output in Jupyter.
from IPython.display import display, Markdown

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [3]:
# ============================================================
# Cell 3: File Configuration
# ============================================================

# Folder containing the PDF documents.
PDF_FOLDER = Path("pdfs")

# Folder where ChromaDB will store the vector database.
CHROMA_PATH = Path("chroma_db")

# Name of the ChromaDB collection.
COLLECTION_NAME = "insurance_rag_collection"

# Sentence Transformer model used to create embeddings.
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# Ollama configuration.
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2"

# RAG configuration.
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200
TOP_K = 5

# Lower temperature makes answers more consistent and grounded.
TEMPERATURE = 0.1

print("Configuration loaded successfully.")
print(f"PDF folder       : {PDF_FOLDER}")
print(f"ChromaDB folder  : {CHROMA_PATH}")
print(f"Embedding model  : {EMBEDDING_MODEL_NAME}")
print(f"Ollama model     : {OLLAMA_MODEL}")

Configuration loaded successfully.
PDF folder       : pdfs
ChromaDB folder  : chroma_db
Embedding model  : all-MiniLM-L6-v2
Ollama model     : llama3.2


In [4]:
# ============================================================
# Cell 4: Verify PDF Documents
# ============================================================

# Find all PDF files inside the pdfs folder.
pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))

# Display the number of PDF files found.
print(f"Number of PDF files found: {len(pdf_files)}")

# Display each PDF filename and its size.
for pdf_file in pdf_files:
    size_mb = pdf_file.stat().st_size / (1024 * 1024)
    print(f"- {pdf_file.name} ({size_mb:.2f} MB)")

# Make sure that at least 3 PDF documents are available.
if len(pdf_files) < 3:
    raise ValueError(
        "At least 3 PDF documents are required for this RAG assignment."
    )

print("\nPDF verification successful.")

Number of PDF files found: 3
- Health_Care_Policy.pdf (0.41 MB)
- Individual_Health_Insurance_policy.pdf (0.62 MB)
- Introduction_To_Insurance_Policy.pdf (7.79 MB)

PDF verification successful.


In [5]:
# ============================================================
# Cell 5: Create Required Project Directories
# ============================================================

# Create the PDF folder if it does not already exist.
PDF_FOLDER.mkdir(parents=True, exist_ok=True)

# Create the ChromaDB folder if it does not already exist.
CHROMA_PATH.mkdir(parents=True, exist_ok=True)

print("Project directories are ready.")
print(f"PDF folder      : {PDF_FOLDER.resolve()}")
print(f"ChromaDB folder : {CHROMA_PATH.resolve()}")

Project directories are ready.
PDF folder      : D:\PDF_RAG_Llama32\pdfs
ChromaDB folder : D:\PDF_RAG_Llama32\chroma_db


In [6]:
# ============================================================
# Cell 6: Extract Text from PDF Documents
# ============================================================

def extract_pdf_pages(pdf_path):
    """
    Extract text from every page of a PDF.

    Each extracted page is stored together with:
    - PDF filename
    - page number
    - page text
    """

    # Open the PDF file.
    reader = PdfReader(str(pdf_path))

    pages = []

    # Process each page.
    for page_number, page in enumerate(reader.pages, start=1):

        # Extract text from the current page.
        text = page.extract_text()

        # Skip pages where no text could be extracted.
        if not text:
            continue

        # Remove unnecessary leading/trailing whitespace.
        text = text.strip()

        # Skip completely empty pages.
        if len(text) == 0:
            continue

        # Store page information.
        pages.append({
            "source": pdf_path.name,
            "page": page_number,
            "text": text
        })

    return pages


# Extract text from all PDF documents.
all_pages = []

for pdf_file in pdf_files:

    print(f"Reading: {pdf_file.name}")

    pages = extract_pdf_pages(pdf_file)

    print(f"  Pages with extracted text: {len(pages)}")

    all_pages.extend(pages)


# Display overall extraction results.
print("\nPDF extraction completed.")
print(f"Total pages with extracted text: {len(all_pages)}")

Reading: Health_Care_Policy.pdf
  Pages with extracted text: 19
Reading: Individual_Health_Insurance_policy.pdf
  Pages with extracted text: 17
Reading: Introduction_To_Insurance_Policy.pdf
  Pages with extracted text: 44

PDF extraction completed.
Total pages with extracted text: 80


In [7]:
# ============================================================
# Cell 7: Inspect Extracted PDF Text
# ============================================================

# Display information from the first few extracted pages.
for i, page_data in enumerate(all_pages[:3], start=1):

    print("=" * 80)
    print(f"DOCUMENT {i}")
    print(f"Source : {page_data['source']}")
    print(f"Page   : {page_data['page']}")
    print("=" * 80)

    # Display the first 1000 characters of the extracted text.
    print(page_data["text"][:1000])

    print("\n")

DOCUMENT 1
Source : Health_Care_Policy.pdf
Page   : 1
UNIVERSAL SOMPO GENERAL INSURANCE CO LTD   
 
Policy Wording – Indian Bank Health Care Plus      UIN: UNIHLIP21044V022021      Page 1 of 19 
INDIAN BANK HEALTH CARE PLUS POLICY 
POLICY WORDING 
 
This policy is an evidence of the contract between you and Universal Sompo General 
Insurance Company Limited. The information furnished by you in the proposal form and the 
declaration signed by you forms the basis of this contract. 
 
The Policy, the Schedule and any Endorsement shall be read together and any word or 
expression to which a specific meaning has been attached in any part of this Policy or of 
Schedule shall bear such meaning whenever it may appear. 
 
This Policy witnesses that in consideration of Your having paid the premium, We undertake 
that if during the period of insurance or during the continuance of this policy by renewal You 
contract any disease or suffer from any illness or sustain any bod ily injury t hrough acc

In [8]:
# ============================================================
# Cell 8: Clean Extracted Text
# ============================================================

def clean_text(text):
    """
    Clean unnecessary whitespace from extracted PDF text.
    """

    # Replace multiple spaces, tabs and line breaks
    # with a single space.
    text = " ".join(text.split())

    return text.strip()


# Create a cleaned copy of all extracted pages.
cleaned_pages = []

for page_data in all_pages:

    cleaned_page = {
        "source": page_data["source"],
        "page": page_data["page"],
        "text": clean_text(page_data["text"])
    }

    cleaned_pages.append(cleaned_page)


# Calculate the total number of characters.
total_characters = sum(
    len(page["text"]) for page in cleaned_pages
)

print("Text cleaning completed.")
print(f"Pages processed       : {len(cleaned_pages)}")
print(f"Total characters      : {total_characters:,}")

Text cleaning completed.
Pages processed       : 80
Total characters      : 168,813


In [13]:
# ============================================================
# Cell 9: Create Word-Boundary-Aware Overlapping Chunks
# ============================================================

def create_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split text into overlapping chunks without breaking words.

    The chunk size is approximately controlled by characters.
    The overlap is preserved between consecutive chunks.
    """

    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:

        # ----------------------------------------------------
        # Find the approximate end of the current chunk.
        # ----------------------------------------------------
        end = min(start + chunk_size, text_length)

        # Move backward until we reach whitespace.
        # This prevents cutting a word in half.
        if end < text_length:
            while end > start and not text[end].isspace():
                end -= 1

        # Extract the current chunk.
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        # Stop if we have reached the end of the document.
        if end >= text_length:
            break

        # ----------------------------------------------------
        # Calculate the beginning of the next chunk.
        # ----------------------------------------------------
        next_start = max(start + 1, end - overlap)

        # Move BACKWARD to the beginning of the word.
        # This preserves the complete word at the overlap.
        while (
            next_start > start
            and not text[next_start - 1].isspace()
        ):
            next_start -= 1

        start = next_start

    return chunks


# ============================================================
# Create chunks while preserving metadata
# ============================================================

chunk_records = []

for page_data in cleaned_pages:

    # Create chunks for the current page.
    page_chunks = create_chunks(page_data["text"])

    # Store each chunk together with its source metadata.
    for chunk_number, chunk_text in enumerate(page_chunks, start=1):

        chunk_records.append({
            "source": page_data["source"],
            "page": page_data["page"],
            "chunk": chunk_number,
            "text": chunk_text
        })


# ============================================================
# Display chunk statistics
# ============================================================

print("Text chunking completed.")
print(f"Pages processed : {len(cleaned_pages)}")
print(f"Total chunks    : {len(chunk_records)}")

if chunk_records:

    average_length = np.mean(
        [len(chunk["text"]) for chunk in chunk_records]
    )

    print(f"Average chunk length: {average_length:.0f} characters")

Text chunking completed.
Pages processed : 80
Total chunks    : 200
Average chunk length: 967 characters


In [15]:
# ============================================================
# Cell 10: Inspect Sample Chunks
# ============================================================

# Display the first three chunks.
for i, chunk in enumerate(chunk_records[:3], start=1):

    print("=" * 80)
    print(f"CHUNK {i}")
    print("=" * 80)

    print(f"Source : {chunk['source']}")
    print(f"Page   : {chunk['page']}")
    print(f"Chunk  : {chunk['chunk']}")

    print("\nText:")
    print(chunk["text"][:800])

    print()

CHUNK 1
Source : Health_Care_Policy.pdf
Page   : 1
Chunk  : 1

Text:
UNIVERSAL SOMPO GENERAL INSURANCE CO LTD Policy Wording – Indian Bank Health Care Plus UIN: UNIHLIP21044V022021 Page 1 of 19 INDIAN BANK HEALTH CARE PLUS POLICY POLICY WORDING This policy is an evidence of the contract between you and Universal Sompo General Insurance Company Limited. The information furnished by you in the proposal form and the declaration signed by you forms the basis of this contract. The Policy, the Schedule and any Endorsement shall be read together and any word or expression to which a specific meaning has been attached in any part of this Policy or of Schedule shall bear such meaning whenever it may appear. This Policy witnesses that in consideration of Your having paid the premium, We undertake that if during the period of insurance or during the continuance of thi

CHUNK 2
Source : Health_Care_Policy.pdf
Page   : 1
Chunk  : 2

Text:
Medical Practitioner, hospitalization for medical/surgical t

In [16]:
# ============================================================
# Cell 11: Load Sentence Transformer Embedding Model
# ============================================================

import os

# Store Hugging Face model files on the D: drive.
# This prevents the embedding model cache from filling C:.
HF_CACHE = Path("huggingface_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE.resolve())
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(HF_CACHE.resolve())

print(f"Hugging Face cache location: {HF_CACHE.resolve()}")

# Load the Sentence Transformer model.
# all-MiniLM-L6-v2 converts text into numerical vectors
# that can be compared using semantic similarity.
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    cache_folder=str(HF_CACHE.resolve())
)

print("\nEmbedding model loaded successfully.")
print(f"Model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

Hugging Face cache location: D:\PDF_RAG_Llama32\huggingface_cache


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\PDF_RAG_Llama32\rag_env\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\PDF_RAG_Llama32\huggingface_cache\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully.
Model: all-MiniLM-L6-v2
Embedding dimension: 384


D:\pip_temp\ipykernel_28896\3206775760.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


In [17]:
# ============================================================
# Cell 12: Generate Embeddings for All Text Chunks
# ============================================================

# Extract the text from every chunk.
documents = [
    chunk["text"]
    for chunk in chunk_records
]

print(f"Generating embeddings for {len(documents)} chunks...")

# Convert every text chunk into a numerical vector.
#
# normalize_embeddings=True normalizes the vectors so that
# cosine similarity can be used effectively for semantic search.
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Convert the embeddings to a NumPy array.
embeddings = np.asarray(embeddings)

print("\nEmbedding generation completed.")
print(f"Number of embeddings : {len(embeddings)}")
print(f"Embedding dimension  : {embeddings.shape[1]}")
print(f"Embedding shape      : {embeddings.shape}")

Generating embeddings for 200 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Embedding generation completed.
Number of embeddings : 200
Embedding dimension  : 384
Embedding shape      : (200, 384)


In [18]:
# ============================================================
# Cell 13: Initialize ChromaDB
# ============================================================

# Create a persistent ChromaDB client.
#
# PersistentClient stores the database on disk so that
# the vector database can be reused between notebook runs.
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH)
)

# Create or retrieve the collection used by our RAG system.
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME
)

print("ChromaDB initialized successfully.")
print(f"Database location : {CHROMA_PATH.resolve()}")
print(f"Collection name   : {COLLECTION_NAME}")
print(f"Existing records  : {collection.count()}")

ChromaDB initialized successfully.
Database location : D:\PDF_RAG_Llama32\chroma_db
Collection name   : insurance_rag_collection
Existing records  : 0


In [19]:
# ============================================================
# Cell 14: Prepare ChromaDB Records
# ============================================================

# Lists required by ChromaDB.
documents = []
metadatas = []
ids = []

# Create a record for every text chunk.
for index, chunk in enumerate(chunk_records):

    # Store the actual chunk text.
    documents.append(chunk["text"])

    # Store metadata that allows us to identify the source.
    metadatas.append({
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk": chunk["chunk"]
    })

    # Create a unique ID for the chunk.
    ids.append(
        f"{Path(chunk['source']).stem}_page_{chunk['page']}_chunk_{chunk['chunk']}_{index}"
    )

print("ChromaDB records prepared.")
print(f"Documents : {len(documents)}")
print(f"Metadata  : {len(metadatas)}")
print(f"IDs       : {len(ids)}")

# Display one example record.
print("\nExample metadata:")
print(metadatas[0])

print("\nExample ID:")
print(ids[0])

ChromaDB records prepared.
Documents : 200
Metadata  : 200
IDs       : 200

Example metadata:
{'source': 'Health_Care_Policy.pdf', 'page': 1, 'chunk': 1}

Example ID:
Health_Care_Policy_page_1_chunk_1_0


In [20]:
# ============================================================
# Cell 15: Store Documents and Embeddings in ChromaDB
# ============================================================

# Convert NumPy embeddings into regular Python lists
# because ChromaDB expects list-based embedding data.
embedding_lists = embeddings.tolist()

# Store the documents, embeddings, metadata and IDs.
collection.upsert(
    documents=documents,
    embeddings=embedding_lists,
    metadatas=metadatas,
    ids=ids
)

print("Documents successfully stored in ChromaDB.")
print(f"Total records in collection: {collection.count()}")

Documents successfully stored in ChromaDB.
Total records in collection: 200


In [21]:
# ============================================================
# Cell 16: Test Semantic Search
# ============================================================

def retrieve_documents(question, top_k=TOP_K):
    """
    Retrieve the most relevant document chunks
    for a given question.
    """

    # Convert the user's question into an embedding.
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    # Search ChromaDB for the most similar chunks.
    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    return results


# ------------------------------------------------------------
# Test question
# ------------------------------------------------------------

test_question = "What expenses are covered for hospitalization?"

results = retrieve_documents(test_question)

print("Question:")
print(test_question)

print("\nRetrieved documents:")
print("=" * 80)

for i, document in enumerate(results["documents"][0], start=1):

    metadata = results["metadatas"][0][i - 1]
    distance = results["distances"][0][i - 1]

    print(f"\nRESULT {i}")
    print("-" * 80)

    print(f"Source   : {metadata['source']}")
    print(f"Page     : {metadata['page']}")
    print(f"Chunk    : {metadata['chunk']}")
    print(f"Distance : {distance:.4f}")

    print("\nText:")
    print(document[:700])

Question:
What expenses are covered for hospitalization?

Retrieved documents:

RESULT 1
--------------------------------------------------------------------------------
Source   : Individual_Health_Insurance_policy.pdf
Page     : 9
Chunk    : 4
Distance : 0.6662

Text:
thereof), plastic surgery except those relating to treatment of Injury or Disease . 6. Cost of spectacles and contact lens or hearing aids. 7. Dental treatment or surgery of any kind. 8. Convalescence, general debility, run down condition or rest cure, congenital external disease or defects or anomalies, sterility, venereal disease, intentional self injury and use of intoxicating drugs/alcohols. 9. Any expense on treatment related to HIV, AIDS and all related medical conditions. 10. Expenses on Diagnostic, X-Ray, or Laboratory examinations unless related to the treatment of Disease or Injury falling within ambit of Hospitalisation or Domiciliary Hospitalisation claim. 11. Expenses on treatme

RESULT 2
------------------

In [22]:
# ============================================================
# Cell 17: Build Context from Retrieved Documents
# ============================================================

def build_context(results):
    """
    Convert retrieved ChromaDB results into a formatted
    context string for the LLM.

    The context contains:
    - Source PDF
    - Page number
    - Chunk number
    - Retrieved text
    """

    context_parts = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    for i, document in enumerate(documents):

        metadata = metadatas[i]

        source = metadata["source"]
        page = metadata["page"]
        chunk_number = metadata["chunk"]

        context_parts.append(
            f"""
--------------------------------------------------
SOURCE: {source}
PAGE: {page}
CHUNK: {chunk_number}
--------------------------------------------------

{document}
"""
        )

    return "\n".join(context_parts)


# Build context using the results from Cell 16.
context = build_context(results)

print("Retrieved context created successfully.")
print(f"Context length: {len(context):,} characters")

print("\nSample of retrieved context:")
print("=" * 80)
print(context[:3000])

Retrieved context created successfully.
Context length: 5,702 characters

Sample of retrieved context:

--------------------------------------------------
SOURCE: Individual_Health_Insurance_policy.pdf
PAGE: 9
CHUNK: 4
--------------------------------------------------

thereof), plastic surgery except those relating to treatment of Injury or Disease . 6. Cost of spectacles and contact lens or hearing aids. 7. Dental treatment or surgery of any kind. 8. Convalescence, general debility, run down condition or rest cure, congenital external disease or defects or anomalies, sterility, venereal disease, intentional self injury and use of intoxicating drugs/alcohols. 9. Any expense on treatment related to HIV, AIDS and all related medical conditions. 10. Expenses on Diagnostic, X-Ray, or Laboratory examinations unless related to the treatment of Disease or Injury falling within ambit of Hospitalisation or Domiciliary Hospitalisation claim. 11. Expenses on treatment arising from or traceable 

In [23]:
# ============================================================
# Cell 18: Create RAG Prompt
# ============================================================

def create_rag_prompt(question, context):
    """
    Create a prompt for Llama 3.2 using the retrieved
    document context.

    The model is instructed to answer only from the
    retrieved insurance documents.
    """

    prompt = f"""
You are an insurance document assistant.

Answer the user's question using ONLY the information
provided in the CONTEXT.

IMPORTANT RULES:

1. Use only the supplied CONTEXT.
2. Do not use outside knowledge.
3. Do not invent facts or information.
4. If the answer cannot be found in the CONTEXT, say:
   "The information was not found in the provided documents."
5. Give a clear and concise answer.
6. When possible, mention the PDF filename and page number.
7. If multiple sources support the answer, mention them.
8. Do not claim that information is present in a document
   unless it appears in the supplied CONTEXT.

==================== CONTEXT ====================

{context}

==================== USER QUESTION ====================

{question}

==================== ANSWER ====================
"""

    return prompt


# Create the RAG prompt using the test question
# and the context retrieved from ChromaDB.
rag_prompt = create_rag_prompt(
    test_question,
    context
)

print("RAG prompt created successfully.")
print(f"Prompt length: {len(rag_prompt):,} characters")

print("\nPrompt preview:")
print("=" * 80)
print(rag_prompt[:2500])

RAG prompt created successfully.
Prompt length: 6,530 characters

Prompt preview:

You are an insurance document assistant.

Answer the user's question using ONLY the information
provided in the CONTEXT.

IMPORTANT RULES:

1. Use only the supplied CONTEXT.
2. Do not use outside knowledge.
3. Do not invent facts or information.
4. If the answer cannot be found in the CONTEXT, say:
   "The information was not found in the provided documents."
5. Give a clear and concise answer.
6. When possible, mention the PDF filename and page number.
7. If multiple sources support the answer, mention them.
8. Do not claim that information is present in a document
   unless it appears in the supplied CONTEXT.

==================== CONTEXT ====================


--------------------------------------------------
SOURCE: Individual_Health_Insurance_policy.pdf
PAGE: 9
CHUNK: 4
--------------------------------------------------

thereof), plastic surgery except those relating to treatment of Injury or Dise

In [24]:
# ============================================================
# Cell 18: Create RAG Prompt
# ============================================================

def create_rag_prompt(question, context):
    """
    Create a prompt for Llama 3.2 using the retrieved
    document context.

    The model is instructed to answer only from the
    retrieved insurance documents.
    """

    prompt = f"""
You are an insurance document assistant.

Answer the user's question using ONLY the information
provided in the CONTEXT.

IMPORTANT RULES:

1. Use only the supplied CONTEXT.
2. Do not use outside knowledge.
3. Do not invent facts or information.
4. If the answer cannot be found in the CONTEXT, say:
   "The information was not found in the provided documents."
5. Give a clear and concise answer.
6. When possible, mention the PDF filename and page number.
7. If multiple sources support the answer, mention them.
8. Do not claim that information is present in a document
   unless it appears in the supplied CONTEXT.

==================== CONTEXT ====================

{context}

==================== USER QUESTION ====================

{question}

==================== ANSWER ====================
"""

    return prompt


# Create the RAG prompt using the test question
# and the context retrieved from ChromaDB.
rag_prompt = create_rag_prompt(
    test_question,
    context
)

print("RAG prompt created successfully.")
print(f"Prompt length: {len(rag_prompt):,} characters")

print("\nPrompt preview:")
print("=" * 80)
print(rag_prompt[:2500])

RAG prompt created successfully.
Prompt length: 6,530 characters

Prompt preview:

You are an insurance document assistant.

Answer the user's question using ONLY the information
provided in the CONTEXT.

IMPORTANT RULES:

1. Use only the supplied CONTEXT.
2. Do not use outside knowledge.
3. Do not invent facts or information.
4. If the answer cannot be found in the CONTEXT, say:
   "The information was not found in the provided documents."
5. Give a clear and concise answer.
6. When possible, mention the PDF filename and page number.
7. If multiple sources support the answer, mention them.
8. Do not claim that information is present in a document
   unless it appears in the supplied CONTEXT.

==================== CONTEXT ====================


--------------------------------------------------
SOURCE: Individual_Health_Insurance_policy.pdf
PAGE: 9
CHUNK: 4
--------------------------------------------------

thereof), plastic surgery except those relating to treatment of Injury or Dise

In [25]:
# ============================================================
# Cell 19: Test Ollama Connection
# ============================================================

def check_ollama():
    """
    Check whether the local Ollama server is running
    and whether the required model is available.
    """

    # Ask Ollama for the list of installed models.
    response = requests.get(
        f"{OLLAMA_BASE_URL}/api/tags",
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    # Extract model names.
    model_names = [
        model["name"]
        for model in data.get("models", [])
    ]

    print("Ollama connection successful.")
    print("\nAvailable models:")

    for model_name in model_names:
        print(f"- {model_name}")

    # Check whether our required model exists.
    model_available = any(
        model_name == OLLAMA_MODEL
        or model_name.startswith(f"{OLLAMA_MODEL}:")
        for model_name in model_names
    )

    if not model_available:
        raise RuntimeError(
            f"{OLLAMA_MODEL} was not found in Ollama."
        )

    print(f"\nRequired model '{OLLAMA_MODEL}' is available.")


# Run the Ollama connection test.
check_ollama()

Ollama connection successful.

Available models:
- llama3.2:latest

Required model 'llama3.2' is available.


In [26]:
# ============================================================
# Cell 20: Generate Response Using Llama 3.2
# ============================================================

def generate_with_ollama(prompt, temperature=TEMPERATURE):
    """
    Send a prompt to Llama 3.2 through the local Ollama API
    and return the generated response.
    """

    # Prepare the request payload.
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    # Send the request to the local Ollama server.
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=payload,
        timeout=300
    )

    # Raise an error if Ollama returns an HTTP error.
    response.raise_for_status()

    # Convert the JSON response into a Python dictionary.
    result = response.json()

    # Return only the generated answer.
    return result["response"].strip()


# ------------------------------------------------------------
# Send our RAG prompt to Llama 3.2.
# ------------------------------------------------------------

print("Sending RAG prompt to Llama 3.2...")
print("Please wait...\n")

answer = generate_with_ollama(rag_prompt)

print("=" * 80)
print("LLAMA 3.2 RESPONSE")
print("=" * 80)
print(answer)

Sending RAG prompt to Llama 3.2...
Please wait...

LLAMA 3.2 RESPONSE
According to the provided documents, the following expenses are covered for hospitalization:

* Room, Boarding, and Nursing Expense as provided in the Hospital/Nursing Home, subject to the following limits:
	+ Normal Room expenses: 1.0% of Basic Sum Insured
	+ Sub limit per day for Intensive Care/ Therapeutic Unit expenses: 2% of Basic Sum Insured
	+ Registration Charges of Hospital/ Nursing Home: Actuals
* Medical Practitioner/ Anesthetist, Consultant fees, Surgeons fees and similar expenses subject to a limit of 25% of Sum Assured
* Expenses on Anesthesia, Blood, Oxygen, Operation Theatre, Surgical Appliances, Medicines and Drugs, Diagnostic Materials and X -ray, Dialysis, Chemotherapy, Radiotherapy, Cost of Pacemaker, Artificial Limbs, Cost of Organs and similar expenses subject to a limit of 40% Sum Insured
* Expenses on Vitamins and Tonics only if forming part of treatment as certified by the attending Medical P

In [27]:
# ============================================================
# Cell 21: Display Answer and Retrieved Sources
# ============================================================

print("=" * 80)
print("RAG ANSWER")
print("=" * 80)

print(answer)

print("\n")
print("=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

# Keep track of sources we have already displayed.
displayed_sources = set()

for metadata in results["metadatas"][0]:

    source = metadata["source"]
    page = metadata["page"]
    chunk = metadata["chunk"]

    source_key = (source, page, chunk)

    if source_key not in displayed_sources:

        print(
            f"- {source} | Page {page} | Chunk {chunk}"
        )

        displayed_sources.add(source_key)

RAG ANSWER
According to the provided documents, the following expenses are covered for hospitalization:

* Room, Boarding, and Nursing Expense as provided in the Hospital/Nursing Home, subject to the following limits:
	+ Normal Room expenses: 1.0% of Basic Sum Insured
	+ Sub limit per day for Intensive Care/ Therapeutic Unit expenses: 2% of Basic Sum Insured
	+ Registration Charges of Hospital/ Nursing Home: Actuals
* Medical Practitioner/ Anesthetist, Consultant fees, Surgeons fees and similar expenses subject to a limit of 25% of Sum Assured
* Expenses on Anesthesia, Blood, Oxygen, Operation Theatre, Surgical Appliances, Medicines and Drugs, Diagnostic Materials and X -ray, Dialysis, Chemotherapy, Radiotherapy, Cost of Pacemaker, Artificial Limbs, Cost of Organs and similar expenses subject to a limit of 40% Sum Insured
* Expenses on Vitamins and Tonics only if forming part of treatment as certified by the attending Medical Practitioner
* Expenses incurred for Domiciliary Hospitaliza

In [28]:
# ============================================================
# Cell 22: Complete RAG Query Pipeline
# ============================================================

def rag_query(question, top_k=TOP_K):
    """
    Complete Retrieval-Augmented Generation pipeline.

    Steps:
        1. Convert the question into an embedding.
        2. Retrieve relevant chunks from ChromaDB.
        3. Build context from the retrieved chunks.
        4. Create the RAG prompt.
        5. Send the prompt to Llama 3.2.
        6. Return the answer and verified sources.
    """

    # --------------------------------------------------------
    # Step 1: Retrieve relevant document chunks
    # --------------------------------------------------------

    retrieved_results = retrieve_documents(
        question,
        top_k=top_k
    )

    # --------------------------------------------------------
    # Step 2: Build context from retrieved chunks
    # --------------------------------------------------------

    retrieved_context = build_context(
        retrieved_results
    )

    # --------------------------------------------------------
    # Step 3: Create the RAG prompt
    # --------------------------------------------------------

    prompt = create_rag_prompt(
        question,
        retrieved_context
    )

    # --------------------------------------------------------
    # Step 4: Generate answer using Llama 3.2
    # --------------------------------------------------------

    generated_answer = generate_with_ollama(
        prompt
    )

    # --------------------------------------------------------
    # Step 5: Extract verified source information
    # --------------------------------------------------------

    sources = []

    for metadata in retrieved_results["metadatas"][0]:

        source = (
            f"{metadata['source']} | "
            f"Page {metadata['page']} | "
            f"Chunk {metadata['chunk']}"
        )

        if source not in sources:
            sources.append(source)

    # --------------------------------------------------------
    # Step 6: Return all RAG results
    # --------------------------------------------------------

    return {
        "question": question,
        "answer": generated_answer,
        "sources": sources,
        "results": retrieved_results,
        "context": retrieved_context
    }


print("Complete RAG pipeline created successfully.")

Complete RAG pipeline created successfully.


In [29]:
# ============================================================
# Cell 23: Test Complete RAG Pipeline
# ============================================================

question = "What are the exclusions mentioned in the health insurance policy?"

# Run the complete RAG pipeline.
rag_result = rag_query(question)

# Display the question.
print("=" * 80)
print("QUESTION")
print("=" * 80)
print(question)

# Display the generated answer.
print("\n")
print("=" * 80)
print("RAG ANSWER")
print("=" * 80)
print(rag_result["answer"])

# Display the verified sources.
print("\n")
print("=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

for source in rag_result["sources"]:
    print(f"- {source}")

QUESTION
What are the exclusions mentioned in the health insurance policy?


RAG ANSWER
The exclusions mentioned in the health insurance policy are:

1. Pre-existing diseases (as per Section 2 of the policy)
2. Hospitalization expenses incurred in the first year of operation of the insurance cover for the following diseases:
    Cataract
    Benign Prostatic Hypertrophy
    Myomectomy, Hysterectomy
    Hernia, Hydrocele
    Fistula in anus, Piles

These exclusions are mentioned in the Health_Care_Policy.pdf document, page 8, chunks 2 and 3.


RETRIEVED SOURCES
- Health_Care_Policy.pdf | Page 8 | Chunk 3
- Individual_Health_Insurance_policy.pdf | Page 8 | Chunk 1
- Health_Care_Policy.pdf | Page 8 | Chunk 2
- Introduction_To_Insurance_Policy.pdf | Page 26 | Chunk 1
- Health_Care_Policy.pdf | Page 5 | Chunk 2


In [30]:
# ============================================================
# Cell 25: Hallucination Prevention Test
# ============================================================

hallucination_question = """
What insurance coverage is provided for underwater
space travel and lunar tourism?
"""

# Run the RAG pipeline.
hallucination_result = rag_query(
    hallucination_question,
    top_k=5
)

# Display the question.
print("=" * 80)
print("HALLUCINATION TEST QUESTION")
print("=" * 80)
print(hallucination_question)

# Display the answer.
print("\n")
print("=" * 80)
print("RAG ANSWER")
print("=" * 80)
print(hallucination_result["answer"])

# Display the retrieved sources.
print("\n")
print("=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

for source in hallucination_result["sources"]:
    print(f"- {source}")

HALLUCINATION TEST QUESTION

What insurance coverage is provided for underwater
space travel and lunar tourism?



RAG ANSWER
The information was not found in the provided documents.


RETRIEVED SOURCES
- Introduction_To_Insurance_Policy.pdf | Page 26 | Chunk 2
- Health_Care_Policy.pdf | Page 9 | Chunk 2
- Health_Care_Policy.pdf | Page 12 | Chunk 3
- Introduction_To_Insurance_Policy.pdf | Page 32 | Chunk 2
- Health_Care_Policy.pdf | Page 10 | Chunk 1


In [36]:
# ============================================================
# Cell 26: Multiple RAG Questions
# ============================================================

test_questions = [
    "What expenses are covered during hospitalization?",
    
    "What are the exclusions mentioned in the health insurance policy?",
    
    "What is domiciliary hospitalization?",
    
    "What conditions are mentioned for hospitalization?",
    
    "What information is provided about pre-hospitalization expenses?"
]


for question_number, question in enumerate(
    test_questions,
    start=1
):

    print("\n")
    print("=" * 80)
    print(f"QUESTION {question_number}")
    print("=" * 80)
    print(question)

    # Run the complete RAG pipeline.
    result = rag_query(question)

    print("\nANSWER")
    print("-" * 80)
    print(result["answer"])

    print("\nSOURCES")
    print("-" * 80)

    for source in result["sources"]:
        print(f"- {source}")



QUESTION 1
What expenses are covered during hospitalization?

ANSWER
--------------------------------------------------------------------------------
The information is found in the Individual_Health_Insurance_policy.pdf, page 7, chunk 2.

According to the policy, the following expenses are covered during hospitalization:

1. Room, Boarding, and Nursing Expenses, subject to the following limits:
	* Normal Room expenses: 1.0% of Basic Sum Insured
	* Sub limit per day for Intensive Care/Therapeutic Unit expenses: 2% of Basic Sum Insured
	* Registration Charges of Hospital/Nursing Home: Actuals
2. Medical Practitioner/Anesthetist, Consultant fees, Surgeons fees and similar expenses, subject to a limit of 25% of Sum Assured.
3. Expenses on Anesthesia, Blood, Oxygen, Operation Theatre, Surgical Appliances, Medicines and Drugs, Diagnostic Materials and X-ray, Dialysis, Chemotherapy, Radiotherapy, Cost of Pacemaker, Artificial Limbs, Cost of Organs and similar expenses, subject to a limit o